# Notebook 2 of 2: training and evaluation

Flagging unanswerable visual questions with image-text fusion. Adelaide University, Multimodal Data Fusion, Project 2.2 (individual submission).

This notebook trains every model in `configs/default.yaml` with the shared training loop in `vizwiz/train.py`, then evaluates them with `vizwiz/evaluate.py`. It reads the features cached by notebook 1, so no photos are needed for training.

| Section | Content |
|---|---|
| 2 | Train all models, 3 seeds each |
| 3 | Main comparison on test: 8 metrics, mean +/- sd over seeds, bootstrap 95% CIs, paired gains |
| 4 | Overfitting: learning curves and train/dev loss gaps |
| 5 | Efficiency: parameters, training time, GPU memory, latency per query |
| 6 | Ablations: soft labels, modality dropout, a second cross-attention layer, CLIP encoders |
| 7 | Robustness: each fusion model with the question or the photo removed |
| 8 | Bias: results by photo sharpness, question frequency and answer type; where errors concentrate; gate values |
| 9 | Calibration and the threshold trade-off for deployment |
| 10 | Failure cases (with example photos) |
| 11 | Numbers for the report, figures and tables saved for GitHub |

**Rules.** Models are trained on train. Early stopping, decision thresholds and the late-fusion weight are chosen on dev. Test is used only for the numbers reported here.

**Run it:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. Training all models takes about 20-30 minutes; finished runs are skipped if you run it again.

## 1. Setup

In [ ]:
import os, sys, json, time, shutil, subprocess, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CONFIG = {
    "repo_url": "https://github.com/Sanjai229/vizwiz-answerability-fusion.git",
    "repo_dir": "/content/vizwiz-answerability-fusion",
    "config": "configs/default.yaml",
    "use_drive": True,
    "drive_dir": "/content/drive/MyDrive/vizwiz_fusion",   # features from notebook 1 are in <drive_dir>/features
    "local_out_dir": "/content/vizwiz_fusion",
    "runs_subdir": "individual_runs",
    "local_features_dir": "/content/features_local",
    "train_args": [],                # extra arguments for vizwiz.train (empty for real runs)
    "n_bootstrap": 1000,
    # Failure-case gallery: downloads the test photos (4 GB) if they are not on the Colab disk already
    "gallery_photos": True,
    "data_dir": "/content/data/vizwiz",
    "test_images_url": "https://vizwiz.cs.colorado.edu/VizWiz_final/images/test.zip",
}

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if CONFIG["use_drive"] and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path(CONFIG["drive_dir"])
else:
    OUT = Path(CONFIG["local_out_dir"])
FEAT_DIR, RUNS_DIR = OUT / "features", OUT / CONFIG["runs_subdir"]
EXPORT = OUT / "results_for_github"
FIGS, TABLES = EXPORT / "figures", EXPORT / "tables"
for d in (RUNS_DIR, FIGS, TABLES):
    d.mkdir(parents=True, exist_ok=True)

REPO = Path(CONFIG["repo_dir"])
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q", CONFIG["repo_url"], str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
sys.path.insert(0, str(REPO))
print(f"Code: {REPO} at commit {COMMIT}")
print(f"Features: {FEAT_DIR}\nRuns: {RUNS_DIR}")


def repo_python(*args):
    # Run `python <args>` from the repo root and print its output as it runs
    proc = subprocess.Popen([sys.executable, *args], cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"python {' '.join(args)} failed (exit code {proc.returncode})")


repo_python("-m", "tests.test_smoke")

## 2. Train every model

One call trains every model in `configs/default.yaml` for seeds 0, 1 and 2, then late fusion and the two floors. The features are first copied from Drive to the Colab disk, which is much faster to read.

| Model | Group | Inputs | Idea |
|---|---|---|---|
| `majority` | floor | none | always predicts the train unanswerable rate |
| `photo_quality` | floor | sharpness, brightness, size | logistic regression: the blur shortcut alone |
| `question_only` | main | BERT mean vector | language priors |
| `image_only` | main | ViT [CLS] vector | photo content |
| `late` | main | the two models above | weighted average of probabilities, weight tuned on dev |
| `early` | main | BERT + ViT vectors joined | one MLP over both |
| `gated` | main | BERT + ViT vectors | Gated Multimodal Unit (Arevalo et al. 2017) with modality dropout |
| `cross_attention` | main | BERT word tokens + ViT 7x7 region tokens | bidirectional cross-attention (Vaswani et al. 2017; Tsai et al. 2019) |
| `*_soft`, `gated_no_modality_dropout`, `cross_attention_2layers` | ablation | as above | one change each |
| `clip_early`, `clip_gated` | reference | CLIP text + image vectors | CLIP encoders (Radford et al. 2021) |

In [ ]:
import yaml

CFG = yaml.safe_load(open(REPO / CONFIG["config"]))
(RUNS_DIR / "_config_used.yaml").write_text(f"# repo commit {COMMIT}\n" + yaml.safe_dump(CFG, sort_keys=False))
t0 = time.time()
repo_python("-m", "vizwiz.train", "--config", CONFIG["config"], "--features-dir", str(FEAT_DIR),
            "--runs-dir", str(RUNS_DIR), "--local-dir", CONFIG["local_features_dir"], *CONFIG["train_args"])
print(f"Training step finished in {(time.time() - t0) / 60:.1f} min")

## 3. Main comparison on the test split

Each run's decision threshold is tuned on dev for macro-F1, then applied to test. The table shows mean +/- standard deviation over the 3 seeds. Metrics: macro-F1 (main metric, treats both classes equally), F1 and recall for the unanswerable class, balanced accuracy, accuracy, AUROC, AUPRC (threshold-free, suits the imbalanced classes; Saito & Rehmsmeier 2015), Brier score and expected calibration error (ECE).

In [ ]:
import importlib
import vizwiz.evaluate as ev
importlib.reload(ev)

meta = pd.read_csv(FEAT_DIR / "meta.csv")
meta["rare_question"] = meta["rare_question"].map({True: "rare (<= 1 in train)", False: "common",
                                                   "True": "rare (<= 1 in train)", "False": "common"})
SPECS = CFG["models"]
NAMES = {"majority": "Majority class", "photo_quality": "Photo quality only", "question_only": "Question only",
         "image_only": "Image only", "late": "Late fusion", "early": "Early fusion", "gated": "Gated fusion",
         "cross_attention": "Cross-attention", "question_only_soft": "Question only, soft labels",
         "image_only_soft": "Image only, soft labels", "gated_no_modality_dropout": "Gated, no modality dropout",
         "cross_attention_2layers": "Cross-attention, 2 layers", "clip_early": "CLIP early fusion",
         "clip_gated": "CLIP gated fusion"}
FLOOR = ["majority", "photo_quality"]
UNIMODAL = ["question_only", "image_only"]
FUSION = ["late", "early", "gated", "cross_attention"]
ABLATION = [m for m in SPECS if SPECS[m].get("group") == "ablation"]
REFERENCE = [m for m in SPECS if SPECS[m].get("group") == "reference"]
ORDER = FLOOR + UNIMODAL + FUSION + ABLATION + REFERENCE
GROUP_OF = {**{m: "floor" for m in FLOOR}, **{m: "single modality" for m in UNIMODAL},
            **{m: "fusion" for m in FUSION}, **{m: "ablation" for m in ABLATION}, **{m: "CLIP reference" for m in REFERENCE}}
name = lambda m: NAMES.get(m, m)

RUNS = ev.load_runs(RUNS_DIR, ORDER)
ORDER = [m for m in ORDER if m in RUNS]
THR = ev.thresholds(RUNS)
SC_TEST, SC_DEV = ev.scores_by_seed(RUNS, THR, "test"), ev.scores_by_seed(RUNS, THR, "dev")
MEAN_TEST, TABLE_TEST = ev.mean_sd_table(SC_TEST, ORDER)
MEAN_DEV, TABLE_DEV = ev.mean_sd_table(SC_DEV, ORDER)
n_test = int((next(iter(RUNS["majority"].values())).split == "test").sum())

show = lambda t: t.rename(index=name)
print(f"Table 1. Test results ({n_test:,} pairs), mean +/- sd over seeds; thresholds tuned on dev")
display(show(TABLE_TEST.loc[[m for m in FLOOR + UNIMODAL + FUSION if m in TABLE_TEST.index]]))
TABLE_TEST.rename(index=name).to_csv(TABLES / "table1_test_results_all_models.csv")
TABLE_DEV.rename(index=name).to_csv(TABLES / "dev_results_all_models.csv")

BEST_UNI = max(UNIMODAL, key=lambda m: MEAN_DEV.loc[m, "macro_f1"])       # chosen on dev
BEST_FUSION = max(FUSION, key=lambda m: MEAN_DEV.loc[m, "macro_f1"])      # chosen on dev
print(f"Best single-modality model (dev): {name(BEST_UNI)}; best fusion model (dev): {name(BEST_FUSION)}")

### Bootstrap confidence intervals and paired gains

The test pairs are resampled 1,000 times (percentile bootstrap; Efron & Tibshirani 1993). For each resample, a model's macro-F1 is averaged over its seeds. All models share the same resamples, so the **paired gain** of one model over another has its own 95% CI. `share_not_better` is the share of resamples where the first model did not beat the second.

In [ ]:
DIST = ev.bootstrap(RUNS, THR, ORDER, n_boot=CONFIG["n_bootstrap"])
CI = ev.ci_table(DIST)
CI_FMT = CI.apply(lambda r: f"{r['mean']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)

pairs = [(m, BEST_UNI) for m in FUSION] + [("cross_attention", "early"), ("gated", "early"), ("early", "late"),
                                            ("clip_early", "early"), ("photo_quality", "majority")]
GAINS = pd.DataFrame([ev.paired_gain(DIST, a, b) for a, b in pairs if a in DIST and b in DIST])
GAINS_SHOW = GAINS.assign(model=GAINS.model.map(name), vs=GAINS.vs.map(name)).round(3)
print("Table 2. Paired bootstrap gains in test macro-F1")
display(GAINS_SHOW)
GAINS_SHOW.to_csv(TABLES / "table2_paired_gains.csv", index=False)
pd.DataFrame({"macro_f1 [95% CI]": CI_FMT}).rename(index=name).to_csv(TABLES / "macro_f1_bootstrap_ci.csv")

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 200, "font.size": 9, "axes.titlesize": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e3",
                     "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.edgecolor": "#8c8c88",
                     "axes.labelcolor": "#2b2b29", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "legend.frameon": False})
COLOUR = {"floor": "#8c8c88", "single modality": "#eb6834", "fusion": "#2a78d6", "ablation": "#6250d6",
          "CLIP reference": "#1baf7a"}


def save(fig, stem):
    fig.savefig(FIGS / f"{stem}.png", bbox_inches="tight", facecolor="white")
    plt.show()


def comparison_figure(models, stem, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 0.42 * len(models) + 1.2), sharey=True)
    ylab = [name(m) for m in models]
    pos = np.arange(len(models))
    col = [COLOUR[GROUP_OF[m]] for m in models]
    ax = axes[0]
    f1 = CI.loc[models, "mean"].values
    err = [f1 - CI.loc[models, "ci_low"].values, CI.loc[models, "ci_high"].values - f1]
    hi = CI.loc[models, "ci_high"].values
    ax.barh(pos, f1, height=0.62, color=col, xerr=err, capsize=3, error_kw={"elinewidth": 1, "ecolor": "#2b2b29"})
    for y, v, h in zip(pos, f1, hi):
        ax.text(h + 0.01, y, f"{v:.3f}", va="center", fontsize=8, color="#2b2b29")
    ax.axvline(CI.loc[BEST_UNI, "mean"], ls=":", lw=1, color=COLOUR["single modality"])
    ax.set(xlabel="Macro-F1 on test (95% bootstrap CI)", xlim=(0.35, min(1.08, hi.max() + 0.08)), title="(a) Macro-F1")
    ax.set_yticks(pos, ylab)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax = axes[1]
    ap = MEAN_TEST.loc[models, "auprc"].values
    sd = SC_TEST.groupby("model")["auprc"].std().reindex(models).fillna(0).values
    ax.barh(pos, ap, height=0.62, color=col, xerr=sd, capsize=3, error_kw={"elinewidth": 1, "ecolor": "#2b2b29"})
    for y, v, e in zip(pos, ap, sd):
        ax.text(v + e + 0.01, y, f"{v:.3f}", va="center", fontsize=8, color="#2b2b29")
    base = float(next(iter(RUNS["majority"].values())).query("split == 'test'").y.mean())
    ax.axvline(base, ls=":", lw=1, color=COLOUR["floor"])
    ax.text(base + 0.005, len(models) - 0.45, "chance", fontsize=7, color="#52514e")
    ax.set(xlabel="AUPRC on test (mean +/- sd over seeds)", xlim=(0.2, min(1.08, float((ap + sd).max()) + 0.08)), title="(b) Area under the precision-recall curve")
    ax.grid(axis="y", visible=False)
    handles = [plt.Rectangle((0, 0), 1, 1, color=COLOUR[g]) for g in dict.fromkeys(GROUP_OF[m] for m in models)]
    fig.legend(handles, list(dict.fromkeys(GROUP_OF[m] for m in models)), loc="lower center",
               ncol=len(handles), bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(title, x=0.01, ha="left", fontsize=11)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    save(fig, stem)


comparison_figure([m for m in FLOOR + UNIMODAL + FUSION if m in RUNS], "fig2_model_comparison",
                  f"Test results ({n_test:,} pairs): floors, single-modality models and fusion models")

## 4. Overfitting

Each epoch, the log loss is measured on 5,000 train pairs (in evaluation mode) and on all dev pairs. A growing gap between them means the model is memorising the train set. Early stopping keeps the epoch with the best dev AUPRC.

In [ ]:
SUMMARIES, HIST = ev.load_summaries(RUNS_DIR), ev.load_histories(RUNS_DIR)
TRAINED = [m for m in ORDER if m in set(HIST.model)]
OVERFIT = ev.overfitting_table(HIST, SUMMARIES, TRAINED)
print("Table 3. Overfitting (mean over seeds): log loss at the selected epoch, and the gap at the last epoch run")
display(OVERFIT.rename(index=name))
OVERFIT.rename(index=name).to_csv(TABLES / "table3_overfitting.csv")

curve_models = [m for m in UNIMODAL + ["early", "gated", "cross_attention"] if m in TRAINED]
fig, axes = plt.subplots(1, len(curve_models), figsize=(3.0 * len(curve_models), 2.8), sharey=True)
for ax, m in zip(np.atleast_1d(axes), curve_models):
    for s, h in HIST[HIST.model == m].groupby("seed"):
        lab = s == 0
        ax.plot(h.epoch, h.train_loss, color="#8c8c88", lw=1.2, alpha=0.9, label="train" if lab else None)
        ax.plot(h.epoch, h.dev_loss, color=COLOUR[GROUP_OF[m]], lw=2, label="dev" if lab else None)
        b = int(h.dev_auprc.values.argmax())
        ax.plot(h.epoch.values[b], h.dev_loss.values[b], "o", ms=6, mfc="white", mec=COLOUR[GROUP_OF[m]], mew=1.5,
                label="selected epoch" if lab else None)
    ax.set(title=name(m), xlabel="Epoch")
np.atleast_1d(axes)[0].set_ylabel("Log loss (3 seeds)")
np.atleast_1d(axes)[0].legend(fontsize=7)
fig.suptitle("Learning curves: train vs dev log loss for each seed (circle = epoch kept by early stopping)", x=0.01, ha="left")
fig.tight_layout()
save(fig, "fig3_learning_curves")

## 5. Efficiency

Cost of each model's fusion head (the frozen encoders are shared by all models). Latency is one forward pass for a single pair on the GPU, with features already extracted. The full cost per query adds the encoders, measured in notebook 1.

In [ ]:
COST = ev.cost_table(SUMMARIES, [m for m in ORDER if m not in FLOOR])
ENC = {}
for f in (OUT / "results" / "encoder_latency.json", OUT / "results" / "data_summary.json"):
    if f.exists():
        d = json.loads(f.read_text())
        ENC = d.get("feasibility", {}).get("encoder_latency", d) if "median_ms" not in d else d
        break
if ENC.get("median_ms") is not None:
    COST["total_ms_per_query"] = (COST["head_latency_ms"] + ENC["median_ms"]).round(1)
print(f"Encoders (ViT-B/16 + BERT-base, one query): {ENC.get('median_ms')} ms median on {ENC.get('device')}")
print("Table 4. Cost per model (mean over seeds)")
display(COST.rename(index=name))
COST.rename(index=name).to_csv(TABLES / "table4_cost.csv")

## 6. Ablations and the CLIP reference

Each ablation changes one thing in a main model. CLIP models use encoders pre-trained to match images and text, so they show what better encoders add on top of the fusion method.

In [ ]:
ABL_PAIRS = [("question_only_soft", "question_only"), ("image_only_soft", "image_only"),
             ("gated_no_modality_dropout", "gated"), ("cross_attention_2layers", "cross_attention"),
             ("clip_early", "early"), ("clip_gated", "gated")]
rows = []
for a, b in ABL_PAIRS:
    if a in MEAN_TEST.index and b in MEAN_TEST.index:
        g = ev.paired_gain(DIST, a, b)
        rows.append({"variant": name(a), "compared with": name(b),
                     **{f"{c}": MEAN_TEST.loc[a, c] for c in ("macro_f1", "auprc", "brier", "ece")},
                     "macro_f1 change": MEAN_TEST.loc[a, "macro_f1"] - MEAN_TEST.loc[b, "macro_f1"],
                     "95% CI": f"[{g['ci_low']:.3f}, {g['ci_high']:.3f}]",
                     "ece change": MEAN_TEST.loc[a, "ece"] - MEAN_TEST.loc[b, "ece"]})
ABLATIONS = pd.DataFrame(rows).round(3)
print("Table 5. Ablations on test (mean over seeds; CI = paired bootstrap of the macro-F1 change)")
display(ABLATIONS)
ABLATIONS.to_csv(TABLES / "table5_ablations.csv", index=False)

## 7. Robustness: what if one input is missing?

Each fusion model is scored again with the question vectors (or the photo vectors) set to zero, keeping the same threshold. A model that falls to chance without the photo relies on the photo; a model trained with modality dropout should degrade more gracefully.

In [ ]:
MISSING = ev.missing_input_table(RUNS, THR, ["early", "gated", "gated_no_modality_dropout", "cross_attention",
                                             "clip_gated"])
print("Table 6. Test scores with one input removed (mean over seeds)")
display(MISSING.rename(index=name))
MISSING.rename(index=name).to_csv(TABLES / "table6_missing_input.csv")

## 8. Bias and subgroups

Test results by photo sharpness group (train tertiles of the variance of the Laplacian), by whether the question is rare in train (seen at most once), and by answer type. Answer type comes from the crowd answers, so it is used here only to describe errors, never as an input. If a model is much better on blurry photos than on sharp ones, it is mostly detecting blur; if it is much worse on rare questions, it relies on memorised phrasing.

In [ ]:
SUB_MODELS = [m for m in ["photo_quality", "question_only", "image_only", "early", "gated", "cross_attention"] if m in RUNS]
SUB = pd.concat([ev.subgroup_table(RUNS, THR, meta, c, SUB_MODELS) for c in ("sharp_bin", "rare_question", "answer_type")])
SUB_WIDE = SUB.pivot_table(index=["group", "level", "n", "unanswerable_rate"], columns="model", values="macro_f1") \
    .reindex(columns=SUB_MODELS).rename(columns=name).round(3)
print("Table 7. Test macro-F1 by subgroup (mean over seeds)")
display(SUB_WIDE)
SUB_WIDE.to_csv(TABLES / "table7_subgroups_macro_f1.csv")
SUB.to_csv(TABLES / "subgroups_long.csv", index=False)

ERR = ev.error_profile(RUNS, THR, meta, BEST_FUSION)
print(f"Where the errors of {name(BEST_FUSION)} concentrate (share of test pairs vs share of errors)")
display(ERR)
ERR.to_csv(TABLES / "error_profile_best_fusion.csv", index=False)

GATE = ev.gate_by_group(RUNS, THR, meta, "gated")
if not GATE.empty:
    print("Gated fusion: mean gate value (weight on the question) by group and true label")
    display(GATE)
    GATE.to_csv(TABLES / "gate_by_group.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), gridspec_kw={"width_ratios": [3, 2, 3]})
levels = {"sharp_bin": ["blurry", "medium", "sharp"], "rare_question": ["common", "rare (<= 1 in train)"],
          "answer_type": sorted(SUB[SUB.group == "answer_type"].level.unique())}
titles = {"sharp_bin": "(a) Photo sharpness", "rare_question": "(b) Question frequency in train",
          "answer_type": "(c) Answer type (description only)"}
plot_models = [m for m in ["photo_quality", "question_only", "image_only", BEST_FUSION] if m in SUB_MODELS]
mcol = {"photo_quality": COLOUR["floor"], "question_only": "#e87ba4", "image_only": COLOUR["single modality"],
        BEST_FUSION: COLOUR["fusion"]}
w = 0.8 / len(plot_models)
for ax, g in zip(axes, levels):
    d = SUB[SUB.group == g]
    x = np.arange(len(levels[g]))
    for i, m in enumerate(plot_models):
        v = d[d.model == m].set_index("level").reindex(levels[g])["macro_f1"].values
        ax.bar(x + (i - (len(plot_models) - 1) / 2) * w, v, width=w * 0.92, color=mcol[m], label=name(m))
    n = d.drop_duplicates("level").set_index("level").reindex(levels[g])
    ax.set_xticks(x, [f"{l}\nn={int(k):,}, {r:.0%} unans." for l, k, r in zip(levels[g], n.n, n.unanswerable_rate)],
                  fontsize=7.5)
    ax.set(title=titles[g], ylim=(0.3, min(1.02, float(np.nanmax(d.macro_f1)) + 0.08)))
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Macro-F1 on test")
axes[0].legend(fontsize=7, loc="upper left", ncol=2)
fig.suptitle("Test macro-F1 by subgroup (mean over seeds)", x=0.01, ha="left")
fig.tight_layout()
save(fig, "fig4_subgroups")

## 9. Calibration and the threshold trade-off

For deployment, the threshold sets how often the assistant asks the user to retake a photo. A false alarm (an answerable question flagged as unanswerable) costs the user time; a missed unanswerable question gives them a wrong answer they cannot check. The table shows, for the best fusion model, the recall of unanswerable questions at the dev-tuned threshold and at thresholds that reach a fixed precision on dev.

In [ ]:
OPS = ev.operating_points(RUNS, THR, BEST_FUSION)
print(f"Table 8. Operating points for {name(BEST_FUSION)} (thresholds chosen on dev, scored on test)")
display(OPS)
OPS.to_csv(TABLES / "table8_operating_points.csv", index=False)
COMP = ev.complementarity(RUNS, THR)
print("Complementarity of the two single-modality models on test:", COMP)

from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
curve = [m for m in ["photo_quality", "question_only", "image_only", "early", BEST_FUSION, "clip_early"] if m in RUNS]
curve = list(dict.fromkeys(curve))
ccol = {"photo_quality": COLOUR["floor"], "question_only": "#e87ba4", "image_only": COLOUR["single modality"],
        "early": "#6250d6", BEST_FUSION: COLOUR["fusion"], "clip_early": COLOUR["CLIP reference"]}
for m in curve:
    e, _ = ev.ensemble(RUNS, THR, m)
    pr, rc, _ = precision_recall_curve(e.y, e.p)
    axes[0].plot(rc, pr, lw=2 if m == BEST_FUSION else 1.4, color=ccol[m], label=f"{name(m)} ({MEAN_TEST.loc[m, 'auprc']:.3f})")
base = e.y.mean()
axes[0].axhline(base, ls=":", lw=1, color="#8c8c88")
axes[0].set(xlabel="Recall (unanswerable)", ylabel="Precision (unanswerable)", title="(a) Precision-recall on test",
            xlim=(0, 1), ylim=(0, 1))
axes[0].legend(fontsize=7, title="AUPRC", title_fontsize=7, loc="upper right")
axes[1].plot([0, 1], [0, 1], ls=":", lw=1, color="#8c8c88")
for m in [m for m in ["question_only_soft", "question_only", "image_only", BEST_FUSION] if m in RUNS]:
    e, _ = ev.ensemble(RUNS, THR, m)
    frac, conf = calibration_curve(e.y, e.p, n_bins=10, strategy="quantile")
    axes[1].plot(conf, frac, marker="o", ms=4, lw=1.5, color=ccol.get(m, COLOUR["ablation"]),
                 label=f"{name(m)} (ECE {MEAN_TEST.loc[m, 'ece']:.3f})")
axes[1].set(xlabel="Predicted P(unanswerable)", ylabel="Observed unanswerable rate", title="(b) Calibration on test",
            xlim=(0, 1), ylim=(0, 1))
axes[1].legend(fontsize=7)
fig.suptitle("Precision-recall and calibration on test (probabilities averaged over seeds)", x=0.01, ha="left")
fig.tight_layout()
save(fig, "fig5_pr_and_calibration")

## 10. Failure cases

The most confident mistakes of the best fusion model (probabilities averaged over its 3 seeds): missed unanswerable questions and false alarms. Example photos are from the VizWiz dataset (CC BY 4.0; Gurari et al. 2018) and are shown only in this notebook and the report, never stored in the repository.

In [ ]:
CASES, COUNTS = ev.failure_cases(RUNS, THR, meta, BEST_FUSION, k=8)
print(f"{name(BEST_FUSION)} on test: {COUNTS['false_negatives']:,} missed unanswerable, "
      f"{COUNTS['false_positives']:,} false alarms (threshold {COUNTS['threshold']:.2f})")
cols = ["error", "pair_id", "question", "p", "sharp_bin", "rare_question", "answer_type", "vote_share"]
display(CASES[cols].round(3))
CASES[["error", "p", "sharp_bin", "rare_question", "answer_type", "vote_share"]].round(3) \
    .to_csv(TABLES / "failure_cases_no_questions.csv", index=False)   # no question text or file names on GitHub

PHOTOS = {}
if CONFIG["gallery_photos"]:
    data = Path(CONFIG["data_dir"])
    data.mkdir(parents=True, exist_ok=True)
    def find_photos():
        return {f: os.path.join(dp, f) for dp, _, fs in os.walk(data) for f in fs if f.startswith("VizWiz_test")}
    PHOTOS = find_photos()
    if not PHOTOS:
        try:
            print("Downloading test photos for the gallery (about 4 GB)...")
            subprocess.run(["wget", "-q", "-O", str(data / "test.zip"), CONFIG["test_images_url"]], check=True)
            subprocess.run(["unzip", "-q", "-o", str(data / "test.zip"), "-d", str(data)], check=True)
            (data / "test.zip").unlink()
            PHOTOS = find_photos()
        except Exception as e:
            print("Could not get the test photos, gallery skipped:", e)

In [ ]:
if PHOTOS and len(CASES):
    from PIL import Image, ImageOps
    show_cases = pd.concat([CASES[CASES.error == "missed unanswerable"].head(4), CASES[CASES.error == "false alarm"].head(4)])
    fig, axes = plt.subplots(2, 4, figsize=(13, 7.2))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), show_cases.iterrows()):
        ax.axis("on")
        path = PHOTOS.get(r.pair_id)
        if path:
            with Image.open(path) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                im.thumbnail((400, 400))
                ax.imshow(im)
        ax.set_xticks([]), ax.set_yticks([])
        ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
        label = "Missed (truly unanswerable)" if r.error == "missed unanswerable" else "False alarm (answerable)"
        ax.set_title(f"{label}\nP(unans.) {r.p:.2f} | {r.sharp_bin} | votes {r.vote_share:.0%}", fontsize=8)
        ax.set_xlabel("\n".join(textwrap.wrap(f"Q: {r.question}", 42)[:3]), fontsize=8)
    fig.suptitle(f"Failure cases of {name(BEST_FUSION)} (photos: VizWiz, CC BY 4.0)", x=0.01, ha="left")
    fig.tight_layout()
    save(fig, "fig6_failure_cases")

## 11. Numbers for the report

Everything printed below is also saved in `results_for_github/summary.json`. The `results_for_github` folder on Drive holds only figures and aggregate tables (no questions or photos), so it can be uploaded to the repository.

In [ ]:
def to_py(o):
    return o.item() if hasattr(o, "item") else str(o)


f = lambda m, c="macro_f1": MEAN_TEST.loc[m, c]
SUMMARY = {"repo_commit": COMMIT, "n_test": n_test, "best_unimodal": BEST_UNI, "best_fusion": BEST_FUSION,
           "test_mean": MEAN_TEST.round(4).reset_index().to_dict(orient="records"),
           "macro_f1_ci": CI.round(4).reset_index().to_dict(orient="records"),
           "paired_gains": GAINS.round(4).to_dict(orient="records"), "ablations": ABLATIONS.to_dict(orient="records"),
           "overfitting": OVERFIT.reset_index().to_dict(orient="records"),
           "cost": COST.reset_index().to_dict(orient="records"), "encoder_latency": ENC,
           "complementarity": COMP, "operating_points": OPS.to_dict(orient="records"), "failure_counts": COUNTS}
(EXPORT / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=to_py))
shutil.copy(RUNS_DIR / "_config_used.yaml", EXPORT / "config_used.yaml")

print("=== Numbers for the report ===")
print(f"Test pairs: {n_test:,}. Best single-modality model (dev): {name(BEST_UNI)}; best fusion model (dev): {name(BEST_FUSION)}.")
for m in [m for m in FLOOR + UNIMODAL + FUSION + REFERENCE if m in MEAN_TEST.index]:
    print(f"{name(m):24s} macro-F1 {CI_FMT[m]}  AUROC {f(m, 'auroc'):.3f}  AUPRC {f(m, 'auprc'):.3f}  "
          f"recall(unans) {f(m, 'recall_unanswerable') if 'recall_unanswerable' in MEAN_TEST else float('nan'):.3f}  "
          f"ECE {f(m, 'ece'):.3f}")
print("\nPaired gains (macro-F1):")
for _, g in GAINS.iterrows():
    print(f"  {name(g.model)} vs {name(g.vs)}: {g.gain:+.3f} [{g.ci_low:+.3f}, {g.ci_high:+.3f}], "
          f"not better in {g.share_not_better:.1%} of resamples")
print("\nAblations (macro-F1 change, ECE change):")
for _, a in ABLATIONS.iterrows():
    print(f"  {a['variant']} vs {a['compared with']}: {a['macro_f1 change']:+.3f} {a['95% CI']}, ECE {a['ece change']:+.3f}")
print("\nOverfitting (gap = dev - train log loss at selected epoch / at last epoch):")
for m, r in OVERFIT.iterrows():
    if m in UNIMODAL + FUSION:
        print(f"  {name(m)}: best epoch {r.best_epoch:.1f} of {r.epochs_run:.1f}, gap {r.gap:+.3f} -> {r.final_gap:+.3f}")
print("\nCost:")
for m, r in COST.iterrows():
    if m in UNIMODAL + FUSION:
        print(f"  {name(m)}: {int(r.params):,} params, {r.sec_per_epoch:.2f} s/epoch, head {r.head_latency_ms:.2f} ms"
              + (f", total {r.total_ms_per_query} ms/query" if "total_ms_per_query" in r else ""))
print("\nMissing input (macro-F1):")
if not MISSING.empty:
    mf = MISSING["macro_f1"]
    for m, r in mf.iterrows():
        print(f"  {name(m)}: both {r.get('both inputs', np.nan):.3f}, no question {r.get('no question', np.nan):.3f}, "
              f"no photo {r.get('no photo', np.nan):.3f}")
print("\nSubgroups (macro-F1):")
for (g, l, n, u), r in SUB_WIDE.iterrows():
    print(f"  {g} = {l} (n={int(n):,}, {u:.1%} unanswerable): " + ", ".join(f"{k} {v:.3f}" for k, v in r.items()))
if not GATE.empty:
    print("\nGate (weight on question):", GATE.to_dict(orient="records"))
print(f"\nComplementarity: exactly one single-modality model right on {COMP['exactly_one_correct']:.1%} of test pairs.")
print(f"Operating points of {name(BEST_FUSION)}:")
for _, r in OPS.iterrows():
    print(f"  {r.rule}: threshold {r.threshold:.2f}, precision {r.precision_unanswerable:.3f}, "
          f"recall {r.recall_unanswerable:.3f}, flags {r.share_flagged:.1%} of questions")
print(f"Failures: {COUNTS}")
print(f"\nSaved figures and tables to {EXPORT}")

## 12. Save a clean copy for GitHub

Colab adds widget data to notebooks, which GitHub cannot display. After **Run all** has finished, this cell saves a clean copy of this notebook (outputs kept, widget data removed) to `notebook_for_github/` on Drive.

In [ ]:
def clean_notebook(nb):
    nb.get("metadata", {}).pop("widgets", None)
    for cell in nb.get("cells", []):
        outs = []
        for o in cell.get("outputs", []):
            data = o.get("data")
            if data is not None:
                data.pop("application/vnd.jupyter.widget-view+json", None)
                if not data:
                    continue
            outs.append(o)
        if "outputs" in cell:
            cell["outputs"] = outs
    return nb


try:
    from google.colab import _message
    nb = _message.blocking_request("get_ipynb", request="", timeout_sec=120)["ipynb"]
    dest = OUT / "notebook_for_github" / "02_train_and_evaluate.ipynb"
    dest.parent.mkdir(exist_ok=True)
    dest.write_text(json.dumps(clean_notebook(nb), indent=1, ensure_ascii=True))
    print("Clean copy saved to", dest)
except Exception as e:
    print("Not in Colab (or the notebook could not be read), so no clean copy was saved:", type(e).__name__)

## References

Arevalo, J, Solorio, T, Montes-y-Gomez, M & Gonzalez, FA 2017, 'Gated multimodal units for information fusion', in *International conference on learning representations (ICLR 2017), workshop track*, Toulon, France.

Baltrusaitis, T, Ahuja, C & Morency, L-P 2019, 'Multimodal machine learning: a survey and taxonomy', *IEEE Transactions on Pattern Analysis and Machine Intelligence*, vol. 41, no. 2, pp. 423-443.

Efron, B & Tibshirani, RJ 1993, *An introduction to the bootstrap*, Chapman & Hall, New York.

Gurari, D, Li, Q, Stangl, AJ, Guo, A, Lin, C, Grauman, K, Luo, J & Bigham, JP 2018, 'VizWiz grand challenge: answering visual questions from blind people', in *Proceedings of the IEEE conference on computer vision and pattern recognition (CVPR)*, IEEE, Salt Lake City, USA, pp. 3608-3617.

Loshchilov, I & Hutter, F 2019, 'Decoupled weight decay regularization', in *International conference on learning representations (ICLR 2019)*, New Orleans, USA.

Radford, A, Kim, JW, Hallacy, C, Ramesh, A, Goh, G, Agarwal, S, Sastry, G, Askell, A, Mishkin, P, Clark, J, Krueger, G & Sutskever, I 2021, 'Learning transferable visual models from natural language supervision', in *Proceedings of the 38th international conference on machine learning (ICML 2021)*, PMLR, vol. 139, pp. 8748-8763.

Saito, T & Rehmsmeier, M 2015, 'The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets', *PLOS ONE*, vol. 10, no. 3, e0118432.

Tsai, Y-HH, Bai, S, Liang, PP, Kolter, JZ, Morency, L-P & Salakhutdinov, R 2019, 'Multimodal transformer for unaligned multimodal language sequences', in *Proceedings of the 57th annual meeting of the Association for Computational Linguistics*, Association for Computational Linguistics, Florence, Italy, pp. 6558-6569.

Vaswani, A, Shazeer, N, Parmar, N, Uszkoreit, J, Jones, L, Gomez, AN, Kaiser, L & Polosukhin, I 2017, 'Attention is all you need', in *Advances in neural information processing systems 30 (NeurIPS 2017)*, Curran Associates, Long Beach, USA, pp. 5998-6008.